# **Cancellation Prediction**

## Objectives

* Extend the preprocessing pipeline with transformations/encoding appropriate for tree-based classification
* Compare candidate classification algorithms using cross-validated recall to identify suitable models
* Conduct hyperparameter tuning on the leading candidate models
* Fit the final classification pipeline (preprocessing + selected model) on the full training set

## Inputs

* Train and test datasets from "outputs/ml_pipeline/preprocessing/" 
* Preprocessing pipeline "outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl"

## Outputs

* Classification preprocessing pipeline saved to "outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl"
* Classification modelling pipeline saved to "outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl"


## Additional Comments

* Model evaluation was moved to a separate notebook for readability; this notebook covers algorithm selection and hyperparameter tuning only.


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

---

## Load Data

In [ ]:
import pandas as pd

X_train = pd.read_csv("outputs/ml_pipeline/preprocessing/X_train.csv")
X_test = pd.read_csv("outputs/ml_pipeline/preprocessing/X_test.csv")
y_train = pd.read_csv("outputs/ml_pipeline/preprocessing/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/ml_pipeline/preprocessing/y_test.csv").squeeze()

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

---

# Add Model Specific Preprocessing

**Classification Pipeline Actions**

| **Feature** | **Prediction model actions** | **Experimental prediction model alternatives** |
| --- | --- | --- |
| is_canceled | Target only | Class weighting; SMOTE |
| lead_time | | Log/skew transformation; binning |
| arrival_date_month | One-hot | Cyclical encoding |
| arrival_date_week_number | Exclude | Cyclical encoding |
|stays_in_weekend_nights||binning|
|stays_in_week_nights||binning|
|adults||binning|
|children||binary, binning|
|babies||binary, binning|
| country | Ordinal encoding | Frequency encoding; target encoding |
| previous_cancellations | | Binary |
| previous_bookings_not_canceled | | Log/skew transformation; binning |
| agent | | Frequency encoding; target encoding; exclude |
| days_in_waiting_list | | Binary; log/skew transformation |
| adr | | Log/skew transformation; binning |
| required_car_parking_spaces | | Binary |
| total_of_special_requests | | Binning; binary |


In [ ]:
drop = ["arrival_date_week_number"]
ordinal = ["country"]
one_hot = ["arrival_date_month"]

* Load and test the preprocessing pipeline

In [ ]:
data = X_train.copy()
data.head(3)

In [ ]:
import joblib
from src.custom_transformers import undefined_meal

preprocessing_pipeline = joblib.load("outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")
pipeline_step1 = preprocessing_pipeline.fit_transform(data)
pipeline_step1.shape


In [ ]:
pipeline_step1.head()

In [ ]:
pipeline_step1["country"].isnull().sum()

* Drop `arrival_date_week_number` 

In [ ]:
from feature_engine.selection import DropFeatures

drop_transformer = DropFeatures(features_to_drop=drop)
pipeline_step2 = drop_transformer.fit_transform(pipeline_step1)
pipeline_step2.shape

* Encode `country`

In [ ]:
from feature_engine.encoding import OrdinalEncoder

encoder = OrdinalEncoder(encoding_method="arbitrary", variables=ordinal)
pipeline_step3 = encoder.fit_transform(pipeline_step2)
pipeline_step3["country"].head(3)

In [ ]:
pipeline_step3["country"].isnull().sum()

* One-hot encode `arrival_date_month`

In [ ]:
from feature_engine.encoding import OneHotEncoder

encoder = OneHotEncoder(variables=one_hot)
pipeline_step4 = encoder.fit_transform(pipeline_step3)
pipeline_step4.shape

* Build the model specific preprocessing pipeline and test

In [ ]:
from feature_engine.encoding import RareLabelEncoder
from sklearn.pipeline import Pipeline

def classification_preprocessing_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", preprocessing_pipeline),
        ("DropFeatures", DropFeatures(features_to_drop=drop)),
        ("RareLabelEncoder", RareLabelEncoder(tol=0.01, variables=ordinal)),
        ("OrdinalEncoder", OrdinalEncoder(encoding_method="arbitrary", variables=ordinal, ignore_format=True)),
        ("OneHotEncoder", OneHotEncoder(variables=one_hot))        
    ])

    return pipeline_base

* Unplanned RareLabelEncoder added because the OrdinalEncoder added NaN values on the unseen test set

In [ ]:
test_df = X_train.copy()
classification_model_preprocessing_pipeline = classification_preprocessing_pipeline()
pipeline_test = classification_model_preprocessing_pipeline.fit_transform(test_df)
pipeline_test.shape

In [ ]:
classification_model_preprocessing_pipeline.named_steps

In [ ]:
pipeline_test["country"].isnull().sum()

* Test that there are no remaining NaN values in `country`

In [ ]:
encoding_error_test = X_test.copy()
error_test = classification_model_preprocessing_pipeline.transform(encoding_error_test)
error_test["country"].isnull().sum()

---

## Classification modelling pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler

def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("Scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline_base

* Model selection

In [ ]:
# Code adapted from the Churnometer walkthrough

from sklearn.model_selection import GridSearchCV
import numpy as np


class ModelComparison:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = classification_pipeline(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

* Use standard hyperparameters to find the most suitable model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

models_search = {
    "LogisticRegression": LogisticRegression(random_state=0, max_iter=500),  # Default value of 100 was insufficient 
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=0),
}

params_search = {
    "LogisticRegression": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "XGBClassifier": {},
    "GradientBoostingClassifier": {},
}

In [ ]:
from sklearn.metrics import make_scorer, recall_score
search = ModelComparison(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

* Multiple algorithms were evaluated using identical processing and 5-fold cross-validation
* Recall was used as the primary optimisation metric because identifying cancellations is the main business requirement
* LogisticRegression was tested to evaluate linear relationships, and to provide a baseline. The task is binary classification so there was little expectation of a strong performance and this was backed up by the results.
* Tree-based methods performed substantially better, with DecisionTree (mean 0.798) and RandomForest (mean 0.797) now the top performers. Neither quite reaches the target recall of 0.8 set out in the business understanding on mean_score, though both exceed it at max_score (0.807 and 0.805 respectively). Given how close both are to target and to each other, these two will be carried forward for tuning and evaluation.
* XGBoost followed closely at 0.789 mean, with a max_score of 0.799, falling just short of target even at its best fold. It remains a reasonable alternative and will continue forward for futher testing.
* std_score for all three tree-based models was comparable (0.006-0.007), suggesting similar stability across folds; none of the models showed erratic fold-to-fold behaviour.
* GradientBoosting and LogisticRegression are dropped at this stage since neither approached the target recall value, with max_scores of 0.707 and 0.612 respectively.

---

## Handle class imbalance 

* Reassess target imbalance

In [ ]:
target = y_train
cxl_df = pd.DataFrame({"Value": target.unique(),
                       "Frequency": target.value_counts().reset_index(drop=True),
                       "Percentage": target.value_counts(normalize=True)})

cxl_df.style.bar(subset=["Percentage"], color="darkgreen").format({"Percentage": "{:.0%}"})

* remove scaling from the pipeline since LogisticRegression has been discounted

In [ ]:
def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", model)
    ])

    return pipeline_base

* Add class weighting to attempt to improve the recall scores of the top 3 performing models

In [ ]:
models_search = {
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
}

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

params_search = {
    "DecisionTreeClassifier": {"model__class_weight": ["balanced"]},
    "RandomForestClassifier": {"model__class_weight": ["balanced", "balanced_subsample"]},
    "XGBClassifier": {"model__scale_pos_weight": [scale_pos_weight]},
}

In [ ]:
search = ModelComparison(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

* Class weighting was applied to address target imbalance ahead of hyperparameter tuning, using `class_weight="balanced"` for DecisionTree and RandomForest, `class_weight="balanced_subsample"` as an additional RandomForest variant, and `scale_pos_weight` (ratio of negative to positive class, ~1.70) for XGBoost
* All configurations were evaluated using the same 5-fold cross-validation setup as the baseline models for direct comparability
* XGBoost with scale_pos_weight showed the strongest improvement, rising from a mean recall of 0.789 to 0.860 - comfortably exceeding the 0.8 target with a tight std_score of 0.002, indicating high stability across folds. This is now the leading candidate.
* RandomForest with class_weight="balanced" also improved substantially, from a mean of 0.797 to 0.839, clearing the target with a reasonably tight std_score of 0.004.
* RandomForest with class_weight="balanced_subsample" showed a much smaller improvement (mean 0.799 vs the unweighted 0.797), suggesting the subsample-based reweighting is diluted relative to the standard "balanced" approach for this dataset. This variant will not be carried forward in favour of the standard balanced weighting.
* DecisionTree with class_weight="balanced" improved to a mean of 0.796, still just short of target, and its std_score increased to 0.009 (vs 0.007 unweighted) - the widest spread of the four configurations, suggesting the class weighting has made this model less stable across folds even though its max_score (0.806) does clear the target.
* Class weighting alone was sufficient to move XGBoost and RandomForest (balanced) well past the 0.8 recall target without requiring resampling (e.g. SMOTE) or threshold adjustment. These two configurations will be carried forward for hyperparameter tuning.

---

## Hyperparameter Tuning

* Create function to compare hyperparameter performance

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

def parameter_comparison(model, param, values):

    results = []

    for value in values:

        current_model = clone(model)
        current_model.set_params(**{param: value})

        scores = cross_val_score(
            classification_pipeline(current_model),
            X_train,
            y_train.values.ravel(),
            scoring="recall",
            cv=5,
            n_jobs=-1,
            verbose=1
        )

        results.append({
            "parameter": param,
            "value": value,
            "recall": scores.mean()
        })

    return pd.DataFrame(results)

**RandomForestClassifier**

In [ ]:
random_forest = RandomForestClassifier(class_weight="balanced", random_state=0)

In [ ]:
max_depth_results = parameter_comparison(
    model=random_forest,
    param="max_depth",
    values=[20, 30, 40]
)
max_depth_results

* Initial test values of `[None, 10, 20]`produced: 0.839, 0.779, 0.845
* Retested with `[20, 30, 40]`, which produced: 0.845, 0.844, 0.840
* Value of 20 will be carried forward

In [ ]:
n_estimators_results = parameter_comparison(
    model=random_forest,
    param="n_estimators",
    values=[100, 300, 500]
)
n_estimators_results

* Initial test values of `[100, 300, 500]` produced insignificant gains in recall for a considerable slow down in test time (10.3s, 32.5s, 56.8s), The gains were not enough to justify the increase in time so the default value of 100 will be retained.

In [ ]:
min_samples_leaf_results = parameter_comparison(
    model=random_forest, 
    param="min_samples_leaf", 
    values=[1, 2, 4])
min_samples_leaf_results

* Initial values tested were `[1, 2, 4]`, producing recall means of 0.839, 0.846 and 0.841
* Value of 2 will be carried forward

**XGBClassifier**

In [ ]:
xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=0)

In [ ]:
xgb_max_depth_results = parameter_comparison(
    model=xgb, 
    param="max_depth", 
    values=[3, 5, 7, 10])
xgb_max_depth_results

* Initial test values of `[3, 5, 7, 10]` resulted in mean recall scores of 0.851, 0.859, 0.860, 0.859
* None of these values improved performance over the default value of 6 which returned 0.8604 in the gridsearch.

In [ ]:
xgb_n_estimators_results = parameter_comparison(
    model=xgb, 
    param="n_estimators", 
    values=[100, 200, 300, 500])
xgb_n_estimators_results

* Inital test values of `[100, 200, 300, 500]` produced mean recall values of 0.860, 0.861, 0.859, 0.852

In [ ]:
xgb_learning_rate_reults = parameter_comparison(
    model=xgb,
    param="learning_rate",
    values = [0.4, 0.5, 0.6]
)
xgb_learning_rate_reults

* Initial test values of `[0.1, 0.2, 0.3, 0.4]` produced mean recall results of 0.849, 0.860, 0.860, 0.862
* Test rerun with values of `[0.4, 0.5, 0.6]` demonstrated that 0.4 was the peak value, with mean recall reducing to 0.859 and 0.856 as `learning_rate` increased.

In [ ]:
xgb_subsample_results = parameter_comparison(
    model=xgb,
    param="subsample",
    values=[0.6, 0.8, 1.0]
)
xgb_subsample_results

* Initial test values of `[0.6, 0.8, 1.0]` revealed that the default value for `subsample` is already optimal

In [ ]:
xgb_colsample_bytree_results = parameter_comparison(
    model=xgb,
    param="colsample_bytree",
    values=[0.6, 0.8, 1.0]
)
xgb_colsample_bytree_results

* Initial test values of `[0.6, 0.8, 1.0]` resulted in mean recall scores of 0.860, 0.862, 0.860 value of 0.8 seems to be the optimum value

* Compare both models with the best performing hyperparameters applied in combination

In [ ]:
models_search = {
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
}

params_search = {
    "RandomForestClassifier": {
        "model__class_weight": ["balanced"],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "XGBClassifier": {
        "model__scale_pos_weight": [scale_pos_weight],
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.2, 0.3, 0.4],
        "model__colsample_bytree": [0.6, 0.8, 1.0]
    }
}

In [ ]:
optimised_models = ModelComparison(models=models_search, params=params_search)
optimised_models.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)

In [ ]:
grid_search_summary, grid_search_pipelines = optimised_models.score_summary(sort_by='mean_score')
grid_search_summary

* From this result I will be taking XGBClassifier forward with `scale_pos_weight=<neg_count/pos_count>`, `learning_rate=0.2` and `n_estimators=200`
* The combined result returned mean recall of 0.863 and it is also the faster of the 2 models tested.

In [ ]:
def classification_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            learning_rate=0.2,
            n_estimators=200,
            random_state=0))
    ])

    return pipeline_base

In [ ]:
X = X_train.copy()
y = y_train.copy()

Xtest = X_test.copy()
ytest = y_test.copy()

classification_model_pipeline = classification_pipeline()
classification_model_pipeline.fit(X, y)

---

## Save Files

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/cancel_predict/v1')
except Exception as e:
  print(e)


* Save prediction preprocessing pipeline

In [ ]:
import joblib

joblib.dump(value=classification_model_preprocessing_pipeline, filename="outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl")

* Save prediction pipeline

In [ ]:
joblib.dump(value=classification_model_pipeline, filename="outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl")